In [ ]:
import sys
import torch
import numpy as np

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy  :", np.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Python : 3.13.15
PyTorch: 2.11.0+cpu
NumPy  : 2.1.3
CUDA   : False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q librosa soundfile scikit-learn matplotlib

Mounted at /content/drive


In [ ]:
import os

BASE_DIR       = "/content/drive/MyDrive/RabieGuard"
FINAL_AUDIO_DIR = os.path.join(BASE_DIR, "datasets", "audio", "final")
MODELS_DIR     = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Audio frontend
SAMPLE_RATE  = 16000
DURATION     = 3
N_MELS       = 64
N_FFT        = 1024
HOP_LENGTH   = 256
TARGET_LEN   = SAMPLE_RATE * DURATION

# Paths for artifacts we will save
AUDIO_MODEL_PATH    = os.path.join(MODELS_DIR, "audio_model_v2.pth")
CALIBRATION_PATH    = os.path.join(MODELS_DIR, "audio_calibration.npz")

# Sanity check: the split folders must exist
for split in ["train", "val", "test"]:
    for label in ["normal", "abnormal"]:
        p = os.path.join(FINAL_AUDIO_DIR, split, label)
        assert os.path.isdir(p), f"MISSING: {p}"
        print(f"OK  {p}  ({len(os.listdir(p))} files)")

print("\nConfig loaded.")

OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/train/normal  (496 files)
OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/train/abnormal  (317 files)
OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/val/normal  (107 files)
OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/val/abnormal  (68 files)
OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/test/normal  (107 files)
OK  /content/drive/MyDrive/RabieGuard/datasets/audio/final/test/abnormal  (69 files)

Config loaded.


In [ ]:
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader

class DogAudioDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        for fname in os.listdir(os.path.join(root_dir, "normal")):
            if fname.lower().endswith((".wav", ".mp3", ".m4a")):
                self.samples.append((os.path.join(root_dir, "normal", fname), 0.0))
        for fname in os.listdir(os.path.join(root_dir, "abnormal")):
            if fname.lower().endswith((".wav", ".mp3", ".m4a")):
                self.samples.append((os.path.join(root_dir, "abnormal", fname), 1.0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)

        if len(audio) < TARGET_LEN:
            audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
        else:
            audio = audio[:TARGET_LEN]

        # Energy feature (dB). Silence-safe via 1e-9 floor.
        rms     = np.sqrt(np.mean(audio ** 2) + 1e-9)
        energy_db = float(20.0 * np.log10(rms + 1e-9))

        mel = librosa.feature.melspectrogram(
            y=audio, sr=SAMPLE_RATE,
            n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)

        return (
            torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0),
            torch.tensor(energy_db, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )


train_ds = DogAudioDataset(os.path.join(FINAL_AUDIO_DIR, "train"))
val_ds   = DogAudioDataset(os.path.join(FINAL_AUDIO_DIR, "val"))
test_ds  = DogAudioDataset(os.path.join(FINAL_AUDIO_DIR, "test"))

BATCH_SIZE = 16

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

mel, eng, lab = next(iter(train_loader))
print("Mel batch :", mel.shape)
print("Energy    :", eng.shape, " sample values:", eng[:4].tolist())
print("Labels    :", lab.shape, " sample values:", lab[:4].tolist())
print("\nDataset and DataLoaders ready.")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Mel batch : torch.Size([16, 1, 64, 188])
Energy    : torch.Size([16])  sample values: [-24.6505126953125, -18.78196907043457, -21.05171012878418, -41.97930908203125]
Labels    : torch.Size([16])  sample values: [1.0, 1.0, 1.0, 1.0]

Dataset and DataLoaders ready.


In [ ]:
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class AudioNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.attention = nn.Sequential(
            nn.Conv2d(64, 1, kernel_size=1),
            nn.Sigmoid(),
        )
        self.flatten_size = 64 * 16 * 47

        self.embed = nn.Sequential(
            nn.Linear(self.flatten_size, 128),
            nn.ReLU(),
        )
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(128, 1)   # <- single logit

    def forward(self, x):
        x = self.features(x)
        x = x * self.attention(x)
        x = x.view(x.size(0), -1)
        emb = self.embed(x)
        logit = self.head(self.dropout(emb)).squeeze(1)
        return logit, emb


model = AudioNet().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"AudioNet v2 created. Params: {n_params:,}")

with torch.no_grad():
    dummy = torch.randn(2, 1, 64, 188).to(DEVICE)
    logit, emb = model(dummy)
    print("Logit shape    :", logit.shape)   # [2]
    print("Embedding shape:", emb.shape)     # [2, 128]

AudioNet v2 created. Params: 6,179,714
Logit shape    : torch.Size([2])
Embedding shape: torch.Size([2, 128])


In [ ]:
import torch.optim as optim

n_normal   = len(os.listdir(os.path.join(FINAL_AUDIO_DIR, "train", "normal")))
n_abnormal = len(os.listdir(os.path.join(FINAL_AUDIO_DIR, "train", "abnormal")))
pos_weight = torch.tensor([n_normal / n_abnormal], dtype=torch.float32).to(DEVICE)

print(f"Train normal  : {n_normal}")
print(f"Train abnormal: {n_abnormal}")
print(f"pos_weight    : {pos_weight.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
NUM_EPOCHS = 20

print("\nTraining configured.")

Train normal  : 496
Train abnormal: 317
pos_weight    : 1.5647

Training configured.


In [ ]:
import time
from sklearn.metrics import roc_auc_score

history = {"train_loss": [], "val_loss": [], "val_auc": []}
best_val_auc = 0.0

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # ---- Train ----
    model.train()
    tr_loss_sum, tr_n = 0.0, 0
    for mel, eng, lab in train_loader:
        mel, lab = mel.to(DEVICE), lab.to(DEVICE)
        optimizer.zero_grad()
        logit, _ = model(mel)
        loss = criterion(logit, lab)
        loss.backward()
        optimizer.step()
        tr_loss_sum += loss.item() * mel.size(0)
        tr_n += mel.size(0)
    tr_loss = tr_loss_sum / tr_n

    # ---- Validate ----
    model.eval()
    val_loss_sum, val_n = 0.0, 0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for mel, eng, lab in val_loader:
            mel, lab = mel.to(DEVICE), lab.to(DEVICE)
            logit, _ = model(mel)
            loss = criterion(logit, lab)
            val_loss_sum += loss.item() * mel.size(0)
            val_n += mel.size(0)
            all_logits.append(logit.cpu())
            all_labels.append(lab.cpu())
    val_loss = val_loss_sum / val_n
    val_logits = torch.cat(all_logits).numpy()
    val_labels = torch.cat(all_labels).numpy()
    val_auc = roc_auc_score(val_labels, val_logits)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    marker = ""
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), AUDIO_MODEL_PATH)
        marker = "  <-- BEST SAVED"

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"tr_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"val_AUC {val_auc:.4f} | {time.time()-t0:.1f}s{marker}")

print(f"\nBest val AUC: {best_val_auc:.4f}")
print("Saved to    :", AUDIO_MODEL_PATH)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2572/1793059598.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 01/20 | tr_loss 1.2958 | val_loss 0.6884 | val_AUC 0.8382 | 101.0s  <-- BEST SAVED


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2572/1793059598.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 02/20 | tr_loss 0.4620 | val_loss 0.5303 | val_AUC 0.8815 | 57.6s  <-- BEST SAVED


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2572/1793059598.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


...

Epoch 18/20 | tr_loss 0.0038 | val_loss 0.7804 | val_AUC 0.9441 | 56.7s


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2572/1793059598.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 19/20 | tr_loss 0.0024 | val_loss 0.8011 | val_AUC 0.9461 | 53.1s


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_2572/1793059598.py:22: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Epoch 20/20 | tr_loss 0.0042 | val_loss 0.7578 | val_AUC 0.9487 | 58.3s

Best val AUC: 0.9512
Saved to    : /content/drive/MyDrive/RabieGuard/models/audio_model_v2.pth


In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score

# Load best checkpoint
model.load_state_dict(torch.load(AUDIO_MODEL_PATH, map_location=DEVICE))
model.eval()

# Collect val logits
val_logits_list, val_labels_list = [], []
with torch.no_grad():
    for mel, eng, lab in val_loader:
        mel = mel.to(DEVICE)
        logit, _ = model(mel)
        val_logits_list.append(logit.cpu())
        val_labels_list.append(lab)
val_logits = torch.cat(val_logits_list).numpy()
val_labels = torch.cat(val_labels_list).numpy()

# Raw probabilities (before calibration)
val_probs_raw = 1.0 / (1.0 + np.exp(-val_logits))

# Fit isotonic regression
iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
iso.fit(val_probs_raw, val_labels)

# Compare raw vs calibrated
val_probs_cal = iso.predict(val_probs_raw)

print(f"Val AUC (raw)         : {roc_auc_score(val_labels, val_probs_raw):.4f}")
print(f"Val AUC (calibrated)  : {roc_auc_score(val_labels, val_probs_cal):.4f}")
print(f"Raw prob range        : [{val_probs_raw.min():.3f}, {val_probs_raw.max():.3f}]")
print(f"Calibrated prob range : [{val_probs_cal.min():.3f}, {val_probs_cal.max():.3f}]")

# Save the calibration curve (the X grid + Y grid define the mapping)
np.savez(
    CALIBRATION_PATH,
    x_thresholds=iso.X_thresholds_,
    y_thresholds=iso.y_thresholds_,
)
print("\nCalibration saved to:", CALIBRATION_PATH)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Val AUC (raw)         : 0.9512
Val AUC (calibrated)  : 0.9579
Raw prob range        : [0.000, 1.000]
Calibrated prob range : [0.000, 1.000]

Calibration saved to: /content/drive/MyDrive/RabieGuard/models/audio_calibration.npz


In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    confusion_matrix,
)

model.eval()
test_logits_list, test_labels_list = [], []
with torch.no_grad():
    for mel, eng, lab in test_loader:
        mel = mel.to(DEVICE)
        logit, _ = model(mel)
        test_logits_list.append(logit.cpu())
        test_labels_list.append(lab)
test_logits = torch.cat(test_logits_list).numpy()
test_labels = torch.cat(test_labels_list).numpy()

# Raw sigmoid
test_probs_raw = 1.0 / (1.0 + np.exp(-test_logits))

# Apply saved calibration
cal = np.load(CALIBRATION_PATH)
iso_loaded = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
iso_loaded.fit(cal["x_thresholds"], cal["y_thresholds"])
test_probs_cal = iso_loaded.predict(test_probs_raw)

# Hard decisions at 0.5
test_preds = (test_probs_cal >= 0.5).astype(int)

print(f"Test AUC              : {roc_auc_score(test_labels, test_probs_cal):.4f}")
print(f"Test accuracy @ 0.5   : {accuracy_score(test_labels, test_preds):.4f}")
print(f"Precision (abnormal)  : {precision_score(test_labels, test_preds):.4f}")
print(f"Recall    (abnormal)  : {recall_score(test_labels, test_preds):.4f}")
print(f"F1        (abnormal)  : {f1_score(test_labels, test_preds):.4f}")

cm = confusion_matrix(test_labels, test_preds)
print("\nConfusion Matrix")
print("              pred_normal  pred_abnormal")
print(f"true_normal    {cm[0,0]:>8d}    {cm[0,1]:>8d}")
print(f"true_abnormal  {cm[1,0]:>8d}    {cm[1,1]:>8d}")

# Distribution check — this is the one that matters for fusion
print("\nCalibrated score distribution")
print(f"  true normal   : mean {test_probs_cal[test_labels==0].mean():.3f}, "
      f"std {test_probs_cal[test_labels==0].std():.3f}")
print(f"  true abnormal : mean {test_probs_cal[test_labels==1].mean():.3f}, "
      f"std {test_probs_cal[test_labels==1].std():.3f}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test AUC              : 0.9252
Test accuracy @ 0.5   : 0.8750
Precision (abnormal)  : 0.9273
Recall    (abnormal)  : 0.7391
F1        (abnormal)  : 0.8226

Confusion Matrix
              pred_normal  pred_abnormal
true_normal         103           4
true_abnormal        18          51

Calibrated score distribution
  true normal   : mean 0.121, std 0.199
  true abnormal : mean 0.759, std 0.342


In [ ]:
def score_audio(clip_path, silence_db_threshold=-50.0):
    """
    Run the audio model on one clip.
    Returns a dict for the fusion layer.
    """
    model.eval()

    audio, _ = librosa.load(clip_path, sr=SAMPLE_RATE, duration=DURATION, mono=True)
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    else:
        audio = audio[:TARGET_LEN]

    rms       = np.sqrt(np.mean(audio ** 2) + 1e-9)
    energy_db = float(20.0 * np.log10(rms + 1e-9))

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)

    mel_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logit, _ = model(mel_tensor)
        logit = logit.item()

    prob_raw = 1.0 / (1.0 + np.exp(-logit))
    prob_cal = float(iso_loaded.predict([prob_raw])[0])

    # Confidence: distance from the decision boundary, scaled to [0,1]
    confidence = float(min(1.0, 2.0 * abs(prob_cal - 0.5)))

    # Presence: is there any meaningful energy in this clip?
    audio_present = energy_db > silence_db_threshold

    return {
        "audio_score":      round(prob_cal, 4),           # 0-1 calibrated
        "audio_risk":       int(round(prob_cal * 100)),   # 0-100
        "audio_confidence": round(confidence, 4),         # 0-1
        "audio_present":    bool(audio_present),
        "energy_db":        round(energy_db, 2),
    }


# ---- Demo on a few test clips ----
import random
random.seed(0)

for label in ["normal", "abnormal"]:
    folder = os.path.join(FINAL_AUDIO_DIR, "test", label)
    sample = random.choice(os.listdir(folder))
    result = score_audio(os.path.join(folder, sample))
    print(f"[{label:8s}] {sample}")
    print(f"    {result}\n")

[normal  ] 00151_rMFxnf5AZr0_20250916_001522_000.wav
    {'audio_score': 0.2273, 'audio_risk': 23, 'audio_confidence': 0.5455, 'audio_present': True, 'energy_db': -20.56}

[abnormal] 00010_2B0DDdu0Nbk_20250915_234044_001.wav
    {'audio_score': 0.9655, 'audio_risk': 97, 'audio_confidence': 0.931, 'audio_present': True, 'energy_db': -22.66}



In [ ]:
import json

contract = {
    "modality": "audio",
    "score_range": [0, 1],
    "risk_range": [0, 100],
    "score_meaning": "calibrated P(abnormal-proxy) — higher = more agitated-sounding",
    "fields": {
        "audio_score":      "float, 0-1, calibrated",
        "audio_risk":       "int, 0-100",
        "audio_confidence": "float, 0-1, distance from decision boundary",
        "audio_present":    "bool, True if energy_db > silence threshold",
        "energy_db":        "float, RMS in dBFS",
    },
    "window": {
        "sample_rate":   SAMPLE_RATE,
        "duration_sec":  DURATION,
        "hop_length":    HOP_LENGTH,
        "n_mels":        N_MELS,
    },
    "model_path":       AUDIO_MODEL_PATH,
    "calibration_path": CALIBRATION_PATH,
    "warning": "trained on proxy data — do not report as rabies detection",
}

contract_path = os.path.join(MODELS_DIR, "audio_fusion_contract.json")
with open(contract_path, "w") as f:
    json.dump(contract, f, indent=2)

print("Contract written to:", contract_path)
print(json.dumps(contract, indent=2))

Contract written to: /content/drive/MyDrive/RabieGuard/models/audio_fusion_contract.json
{
  "modality": "audio",
  "score_range": [
    0,
    1
  ],
  "risk_range": [
    0,
    100
  ],
  "score_meaning": "calibrated P(abnormal-proxy) \u2014 higher = more agitated-sounding",
  "fields": {
    "audio_score": "float, 0-1, calibrated",
    "audio_risk": "int, 0-100",
    "audio_confidence": "float, 0-1, distance from decision boundary",
    "audio_present": "bool, True if energy_db > silence threshold",
    "energy_db": "float, RMS in dBFS"
  },
  "window": {
    "sample_rate": 16000,
    "duration_sec": 3,
    "hop_length": 256,
    "n_mels": 64
  },
  "model_path": "/content/drive/MyDrive/RabieGuard/models/audio_model_v2.pth",
  "calibration_path": "/content/drive/MyDrive/RabieGuard/models/audio_calibration.npz",
  "warning": "trained on proxy data \u2014 do not report as rabies detection"
}


In [ ]:
import os, json, shutil, datetime

# ============================================================
# 1. MODEL BACKUP
# ============================================================

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Primary model path (set in Cell 3)
# AUDIO_MODEL_PATH = .../models/audio_model_v2.pth

# Backup with timestamp so you can never lose it
backup_path = os.path.join(
    MODELS_DIR, f"audio_model_v2_{timestamp}.pth"
)
shutil.copy2(AUDIO_MODEL_PATH, backup_path)

# Also make sure the calibration file is backed up
cal_backup = os.path.join(
    MODELS_DIR, f"audio_calibration_{timestamp}.npz"
)
shutil.copy2(CALIBRATION_PATH, cal_backup)

print("Model saved")
print("  primary :", AUDIO_MODEL_PATH)
print("  backup  :", backup_path)
print("Calibration saved")
print("  primary :", CALIBRATION_PATH)
print("  backup  :", cal_backup)


# ============================================================
# 2. FUSION STRATEGY DOCUMENT
# ============================================================

fusion_strategy = {
    "version": "1.0",
    "date": timestamp,
    "title": "Audio-Video Fusion Strategy (Interim, Proxy-Data Stage)",

    "caveat": (
        "Neither modality is detecting rabies at this stage. "
        "Both models are trained on proxy data and output a "
        "'agitation-likeness' score. Do NOT describe these outputs "
        "as rabies detection in any report, slide, or conversation."
    ),

    "score_definitions": {
        "audio_score": {
            "range": [0.0, 1.0],
            "meaning": "calibrated P(agitated-proxy | audio)",
            "source": "AudioNet v2 + isotonic calibration",
        },
        "video_score": {
            "range": [0.0, 1.0],
            "meaning": "calibrated P(agitated-proxy | video)",
            "source": "VideoNet (owned by video handler)",
        },
    },

    "reliability_signals": {
        "audio_reliability": {
            "description": "how trustworthy the audio score is, based on raw signal",
            "components": [
                "audio_present (bool)  — energy_db > silence threshold",
                "energy_db (float)     — RMS in dBFS",
                "snr_estimate (float)  — optional, if computed",
            ],
            "range": [0.0, 1.0],
            "fallback": "0.0 when audio_present is False",
        },
        "video_reliability": {
            "description": "how trustworthy the video score is, based on raw signal",
            "components": [
                "mean_luma (float)     — brightness of the frame",
                "motion_energy (float)",
                "video_present (bool)  — e.g. luma > darkness threshold",
            ],
            "range": [0.0, 1.0],
            "fallback": "0.0 when video_present is False",
        },
    },

    "confidence_signals": {
        "audio_confidence": "min(1, 2 * |audio_score - 0.5|)",
        "video_confidence": "same formula on video_score",
        "purpose": "distinguishes 'I am sure this is agitated' from 'I have no idea'",
    },

    "fusion_rule": {
        "type": "reliability-and-confidence-weighted average",
        "formula": (
            "w_audio  = audio_reliability  * audio_confidence\n"
            "w_video  = video_reliability  * video_confidence\n"
            "final    = (w_audio * audio_score + w_video * video_score) "
            "/ (w_audio + w_video + eps)"
        ),
        "eps": 1e-8,
        "notes": [
            "When audio_present is False → w_audio = 0 → final falls back to video-only.",
            "When video is dark → video_reliability ≈ 0 → final falls back to audio-only.",
            "When both are weak → both weights near 0 → fusion emits low confidence, not a random score.",
        ],
    },

    "temporal_alignment": {
        "fusion_tick_hz": 2,
        "tick_seconds": 0.5,
        "time_origin": "t=0 at first video frame of the clip",
        "audio_window_seconds": 3.0,
        "audio_hop_seconds": 1.5,
        "rule": (
            "Both modalities emit one (score, reliability, confidence) "
            "triple per fusion tick. A modality's internal window may span "
            "several ticks; the value for each tick is the average over the "
            "windows overlapping that tick."
        ),
    },

    "missing_modality_contract": {
        "present_field": "explicit boolean, never a numeric sentinel",
        "missing_audio": "audio_present=False, audio_score=None",
        "missing_video": "video_present=False, video_score=None",
        "both_missing": "fusion emits final=None, no_score_reason='no_input'",
    },

    "per_tick_payload": {
        "t":                 "float, seconds from clip start",
        "audio_score":       "float | None",
        "audio_reliability": "float | None",
        "audio_confidence":  "float | None",
        "audio_present":     "bool",
        "video_score":       "float | None",
        "video_reliability": "float | None",
        "video_confidence":  "float | None",
        "video_present":     "bool",
        "final_score":       "float | None  — 0..1, calibrated",
        "final_risk":        "int | None    — 0..100",
        "final_confidence":  "float | None",
    },

    "deployment_notes": [
        "Darkness threshold and silence threshold must be agreed in writing.",
        "Both sides must resample onto the same 2 Hz grid before fusion.",
        "Calibrate each modality separately, then fuse — do not fuse raw logits.",
        "Log both raw and calibrated scores during testing for debugging.",
    ],
}


# ============================================================
# 3. WRITE DOCUMENT
# ============================================================

doc_dir = os.path.join(BASE_DIR, "docs")
os.makedirs(doc_dir, exist_ok=True)

# Machine-readable JSON (for the fusion code to load)
json_path = os.path.join(doc_dir, "fusion_strategy.json")
with open(json_path, "w") as f:
    json.dump(fusion_strategy, f, indent=2)

# Human-readable Markdown (for the video team)
md_lines = []
md_lines.append(f"# {fusion_strategy['title']}")
md_lines.append(f"\n_Version {fusion_strategy['version']} — {fusion_strategy['date']}_\n")
md_lines.append(f"> **{fusion_strategy['caveat']}**\n")

md_lines.append("## Score Definitions\n")
for k, v in fusion_strategy["score_definitions"].items():
    md_lines.append(f"- **{k}**  ({v['range'][0]}–{v['range'][1]}): {v['meaning']}  \n  Source: `{v['source']}`")

md_lines.append("\n## Reliability Signals\n")
for k, v in fusion_strategy["reliability_signals"].items():
    md_lines.append(f"- **{k}**: {v['description']}")
    for c in v["components"]:
        md_lines.append(f"    - {c}")

md_lines.append("\n## Confidence Signals\n")
for k, v in fusion_strategy["confidence_signals"].items():
    md_lines.append(f"- **{k}**: {v}")

md_lines.append("\n## Fusion Rule\n")
md_lines.append(f"Type: **{fusion_strategy['fusion_rule']['type']}**\n")
md_lines.append("```")
md_lines.append(fusion_strategy["fusion_rule"]["formula"])
md_lines.append("```")
for n in fusion_strategy["fusion_rule"]["notes"]:
    md_lines.append(f"- {n}")

md_lines.append("\n## Temporal Alignment\n")
for k, v in fusion_strategy["temporal_alignment"].items():
    md_lines.append(f"- **{k}**: {v}")

md_lines.append("\n## Missing-Modality Contract\n")
for k, v in fusion_strategy["missing_modality_contract"].items():
    md_lines.append(f"- **{k}**: {v}")

md_lines.append("\n## Per-Tick Payload\n")
md_lines.append("| Field | Description |")
md_lines.append("|---|---|")
for k, v in fusion_strategy["per_tick_payload"].items():
    md_lines.append(f"| `{k}` | {v} |")

md_lines.append("\n## Deployment Notes\n")
for n in fusion_strategy["deployment_notes"]:
    md_lines.append(f"- {n}")

md_path = os.path.join(doc_dir, "fusion_strategy.md")
with open(md_path, "w") as f:
    f.write("\n".join(md_lines))


# ============================================================
# 4. VIDEO TEAM MESSAGE
# ============================================================

video_team_message = f"""
# Message to Video Team — Audio Side Ready

Hi,

The audio branch of the pipeline is now trained and its fusion
interface is frozen. Summary:

- Audio model: AudioNet v2 (mel-spectrogram CNN, single-logit head).
- Output: calibrated P(agitated-proxy | audio), scaled to 0–100.
- Reliability + confidence signals emitted per window (see below).
- Calibration object saved alongside the model.

Files (all under `{MODELS_DIR}`):
- `{os.path.basename(AUDIO_MODEL_PATH)}`         — model weights
- `{os.path.basename(CALIBRATION_PATH)}` — isotonic calibration curves
- `{os.path.basename(backup_path)}` — backup, timestamped
- `{os.path.basename(cal_backup)}` — calibration backup, timestamped

Full contract and fusion strategy:
- `{json_path}`
- `{md_path}`

## What I emit per window

Every {fusion_strategy['temporal_alignment']['audio_window_seconds']} s
(window), with a hop of
{fusion_strategy['temporal_alignment']['audio_hop_seconds']} s:

    {{
      "t_start":          float,
      "t_end":            float,
      "audio_score":      float | None,   # 0-1, calibrated
      "audio_risk":       int   | None,   # 0-100
      "audio_reliability":float | None,   # 0-1
      "audio_confidence": float | None,   # 0-1
      "audio_present":    bool,
      "energy_db":        float,
    }}

`audio_present = False` means the window is silent — reliability
drops to 0 and fusion should fall back to video-only.

## What I need from you

1. Your `video_score`, `video_reliability`, `video_confidence`,
   `video_present` on the same time grid — one triple per
   {fusion_strategy['temporal_alignment']['tick_seconds']} s tick.
2. Agreement on the darkness threshold (I suggest mean luma < 40/255).
3. Agreement on the silence threshold (I am using RMS < -50 dBFS).
4. Confirmation of the time origin — I assume t=0 is the first video frame.

## Fusion rule (Block 5, owned by you)

    w_audio = audio_reliability * audio_confidence
    w_video = video_reliability * video_confidence
    final   = (w_audio * audio_score + w_video * video_score)
              / (w_audio + w_video + 1e-8)

This gives graceful fallback: dark video → video weight ≈ 0,
silent audio → audio weight ≈ 0.

## Reminder

Both scores are calibrated probabilities of an *agitation proxy*.
Neither model detects rabies. Please keep this wording in any
shared notes or slides — the project doc is strict about it.

Ping me when your side emits the same tick payload and we can
test fusion on the Bioacoustic dataset (Zenodo 10.5281/zenodo.18972388).

— audio side
"""

msg_path = os.path.join(doc_dir, "message_to_video_team.md")
with open(msg_path, "w") as f:
    f.write(video_team_message)


# ============================================================
# 5. CONFIRMATION
# ============================================================

print("\n" + "=" * 60)
print("SAVED ARTIFACTS")
print("=" * 60)
for label, path in [
    ("model primary",    AUDIO_MODEL_PATH),
    ("model backup",     backup_path),
    ("calibration",      CALIBRATION_PATH),
    ("calibration bkp",  cal_backup),
    ("fusion JSON",      json_path),
    ("fusion MD",        md_path),
    ("video team msg",   msg_path),
]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    print(f"{label:18s} | {'OK ' if exists else 'MISSING'} | {size:>10d} bytes | {path}")

Model saved
  primary : /content/drive/MyDrive/RabieGuard/models/audio_model_v2.pth
  backup  : /content/drive/MyDrive/RabieGuard/models/audio_model_v2_20260923_181339.pth
Calibration saved
  primary : /content/drive/MyDrive/RabieGuard/models/audio_calibration.npz
  backup  : /content/drive/MyDrive/RabieGuard/models/audio_calibration_20260923_181339.npz

SAVED ARTIFACTS
model primary      | OK  |   24726941 bytes | /content/drive/MyDrive/RabieGuard/models/audio_model_v2.pth
model backup       | OK  |   24726941 bytes | /content/drive/MyDrive/RabieGuard/models/audio_model_v2_20260923_181339.pth
calibration        | OK  |        694 bytes | /content/drive/MyDrive/RabieGuard/models/audio_calibration.npz
calibration bkp    | OK  |        694 bytes | /content/drive/MyDrive/RabieGuard/models/audio_calibration_20260923_181339.npz
fusion JSON        | OK  |       4073 bytes | /content/drive/MyDrive/RabieGuard/docs/fusion_strategy.json
fusion MD          | OK  |       3322 bytes | /content/driv

In [ ]:
# ---- Simulate a fresh session ----
import os, numpy as np, torch, librosa
from sklearn.isotonic import IsotonicRegression
import torch.nn as nn

# Rebuild the exact same architecture class (must match training)
class AudioNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.attention = nn.Sequential(nn.Conv2d(64, 1, 1), nn.Sigmoid())
        self.flatten_size = 64 * 16 * 47
        self.embed = nn.Sequential(nn.Linear(self.flatten_size, 128), nn.ReLU())
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = x * self.attention(x)
        x = x.view(x.size(0), -1)
        emb = self.embed(x)
        return self.head(self.dropout(emb)).squeeze(1), emb


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reloaded = AudioNet().to(DEVICE)
reloaded.load_state_dict(torch.load(AUDIO_MODEL_PATH, map_location=DEVICE))
reloaded.eval()

# Reload calibration
cal = np.load(CALIBRATION_PATH)
iso_reload = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
iso_reload.fit(cal["x_thresholds"], cal["y_thresholds"])

print("Model reloaded from disk")
print("Calibration reloaded from disk")

# Smoke test on one clip
import random
folder = os.path.join(FINAL_AUDIO_DIR, "test", "abnormal")
sample = random.choice(os.listdir(folder))
path   = os.path.join(folder, sample)

audio, _ = librosa.load(path, sr=16000, duration=3, mono=True)
if len(audio) < 48000:
    audio = np.pad(audio, (0, 48000 - len(audio)))
else:
    audio = audio[:48000]

mel = librosa.feature.melspectrogram(
    y=audio, sr=16000, n_fft=1024, hop_length=256, n_mels=64
)
mel = librosa.power_to_db(mel, ref=np.max)
mel = (mel - mel.mean()) / (mel.std() + 1e-8)

with torch.no_grad():
    t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    logit, _ = reloaded(t)
    logit = logit.item()

prob = 1.0 / (1.0 + np.exp(-logit))
cal_prob = float(iso_reload.predict([prob])[0])

print(f"\nSmoke test on: {sample}")
print(f"  raw logit    : {logit:+.4f}")
print(f"  sigmoid      : {prob:.4f}")
print(f"  calibrated   : {cal_prob:.4f}")
print(f"  risk (0-100) : {int(round(cal_prob * 100))}")
print("\nReload path works. You can now resume without retraining.")

Model reloaded from disk
Calibration reloaded from disk

Smoke test on: 00168_Doggrowl98_1.wav
  raw logit    : +3.7133
  sigmoid      : 0.9762
  calibrated   : 0.7309
  risk (0-100) : 73

Reload path works. You can now resume without retraining.
